This notebook performs inference on a couple of images and then produces an
example image grid.


In [ ]:
from pathlib import Path

import torch
from torchvision.io import read_image
from torchvision.utils import draw_bounding_boxes

from colony_segmentation.dataset_transforms import get_transforms
from colony_segmentation.model import get_model_instance_segmentation


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")

eval_transform = get_transforms(is_train=False)


def draw_boxes_on_image(image: torch.Tensor, boxes: torch.Tensor):
    image = draw_bounding_boxes(
        image,
        boxes,
        colors="#ff1111",
        width=8,
        font_size=52,
        font=str(Path("./assets/zed-mono-bold.ttf")),
    )

    return image


model = get_model_instance_segmentation(2).to(device)
model.load_state_dict(
    torch.load(
        "colony_segmentation/artifacts/2024-06-05_16-12/final.pth", map_location=device
    )
)


def predict(image: torch.Tensor):
    model.eval()
    with torch.no_grad():
        x = eval_transform(image)
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    return image, pred["boxes"].long(), pred["scores"]

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Specify the path
image_path = Path("data/unlabeled")

examples = [
    image_path / "default_mmrsbpb_default_19012024135915.jpg",
    image_path / "default_mmrsbpb_default_19012024140213.jpg",
    image_path / "default_mmrsbpb_default_19012024140729.jpg",
    image_path / "default_mmrsbpb_default_19012024140822.jpg",
    image_path / "default_mmrsbpb_default_19012024140839.jpg",
    image_path / "default_mmrsbpb_default_19012024141941.jpg",
    image_path / "default_mmrsbpb_default_19012024142454.jpg",
    image_path / "default_mmrsbpb_default_19012024142706.jpg",
    image_path / "default_mmrsbpb_default_19012024142935.jpg",
    image_path / "default_mmrsbpb_default_19012024142951.jpg",
    image_path / "default_mmrsbpb_default_19012024143336.jpg",
    image_path / "default_mmrsbpb_default_19012024143353.jpg",
    image_path / "default_mmrsbpb_default_19012024144350.jpg",
    image_path / "default_mmrsbpb_default_19012024144405.jpg",
]

# Set up the figure and axes
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(9, 12))

# Loop through the files and plot them
for ax, jpg_file in tqdm(zip(axes.flat, examples)):
    image = read_image(jpg_file)

    _, pred_boxes, __ = predict(image)
    pred_image = draw_boxes_on_image(image, pred_boxes)

    ax.imshow(pred_image.permute(1, 2, 0).numpy())
    ax.axis("off")

# Remove spacing between subplots
plt.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0, hspace=0)
plt.savefig("chart_example_predictions.jpg", dpi=500)
plt.show()